Instalación de librerías para RAG local

In [ ]:

%pip install -q langchain langchain-community langchain-huggingface sentence_transformers chromadb langchain-chroma
%pip install torch --force-reinstall --index-url https://download.pytorch.org/whl/cu121
%pip install -q pypdf


print("¡Librerías importadas correctamente!")

Crear o actualizar la base de datos

In [ ]:
import os
import torch
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

directorio_pdfs = "./leyes_pdf"
persist_dir = "./chroma_db_boe_completo_save"

print(f"Cargando base de datos vectorial existente en '{persist_dir}'...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'batch_size': 256, 'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Obtener metadatos de Chroma
datos_existentes = vectorstore.get(include=['metadatas'])
archivos_indexados = {}  # { ruta_normalizada: mtime_almacenado }

if datos_existentes and datos_existentes.get('metadatas'):
    for meta in datos_existentes['metadatas']:
        if meta and 'source' in meta:
            ruta = os.path.normpath(meta['source'])
            mtime = meta.get('mtime', 0)
            archivos_indexados[ruta] = mtime

print(f"Archivos únicos ya registrados en la BD: {len(archivos_indexados)}")

# Cargar todos los nuevos PDF
loader = PyPDFDirectoryLoader(directorio_pdfs)
todos_los_documentos = loader.load()

# Detectar archivos nuevos o modificados
documentos_a_procesar = []
archivos_a_eliminar = set()

for doc in todos_los_documentos:
    ruta_doc = os.path.normpath(doc.metadata.get('source', ''))
    
    if os.path.exists(ruta_doc):
        mtime_actual = os.path.getmtime(ruta_doc)
      
        doc.metadata['mtime'] = mtime_actual
        
        if ruta_doc not in archivos_indexados:
            # Archivo nuevo
            documentos_a_procesar.append(doc)
        elif mtime_actual > archivos_indexados[ruta_doc]:
            # Archivo modificado
            documentos_a_procesar.append(doc)
            archivos_a_eliminar.add(ruta_doc)

if not documentos_a_procesar:
    print("¡No hay PDFs nuevos ni modificados! La base de datos está completamente al día.")
else:
    # Borrar desactualizados
    if archivos_a_eliminar:
        print(f"Eliminando fragmentos desactualizados de {len(archivos_a_eliminar)} archivo(s) reescrito(s)...")
        for ruta_borrar in archivos_a_eliminar:
            # Chroma permite filtrar borrados por metadata
            vectorstore.delete(where={"source": ruta_borrar})

    archivos_unicos_nuevos = set(doc.metadata.get('source') for doc in documentos_a_procesar)
    print(f"Procesando {len(archivos_unicos_nuevos)} archivo(s) ({len(documentos_a_procesar)} páginas).")

    # Fragmentar e indexar por lotes
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks_nuevos = text_splitter.split_documents(documentos_a_procesar)
    total_chunks_nuevos = len(chunks_nuevos)
    
    batch_size_chroma = 4000
    print("Indexando fragmentos actualizados en ChromaDB...")
    
    for i in range(0, total_chunks_nuevos, batch_size_chroma):
        lote = chunks_nuevos[i:i + batch_size_chroma]
        vectorstore.add_documents(documents=lote)
        print(f"  -> Procesados {min(i + batch_size_chroma, total_chunks_nuevos)} / {total_chunks_nuevos} fragmentos...")

    print("¡Base de datos actualizada e indexada correctamente!")

C:\Users\mroscardd\AppData\Local\Temp\ipykernel_17372\3419128458.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\mroscardd\Desktop\Code\IA Keepcoding\proyectos\Qlora_1\BOE_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando base de datos vectorial existente en './chroma_db_boe_completo_save'...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18590.28it/s]


Archivos únicos ya registrados en la BD: 2
Eliminando fragmentos desactualizados de 2 archivo(s) reescrito(s)...
Procesando 2 archivo(s) (237 páginas).
Indexando fragmentos actualizados en ChromaDB...
  -> Procesados 1016 / 1016 fragmentos...
¡Base de datos actualizada e indexada correctamente!
